In [1]:
import cv2
import numpy as np
import os
import face_recognition as fr

In [2]:
path="attendance"

In [3]:
os.listdir(path)

['adarsh_madhu.jpeg',
 'anagha.jpeg',
 'anooja_giri.jpeg',
 'arjun_jayappan.jpeg',
 'celcia_george.jpeg',
 'jomol_sabu.jpeg']

In [4]:
mylist=os.listdir(path)
mylist

['adarsh_madhu.jpeg',
 'anagha.jpeg',
 'anooja_giri.jpeg',
 'arjun_jayappan.jpeg',
 'celcia_george.jpeg',
 'jomol_sabu.jpeg']

In [5]:
imgs=[]
classnames=[]
for i in mylist:
    imgpath=os.path.join(path,i)
    img1=cv2.imread(imgpath)
    imgs.append(img1)
    classnames.append(i.split(".")[0]) 
classnames

['adarsh_madhu',
 'anagha',
 'anooja_giri',
 'arjun_jayappan',
 'celcia_george',
 'jomol_sabu']

In [6]:
def faceencodings(attendance):
    encodelist=[]
    for i in imgs:
        img1=cv2.cvtColor(i,cv2.COLOR_BGR2RGB)
        face_in_frame=fr.face_locations(img1)
        face_encode=fr.face_encodings(img1,face_in_frame)[0]
        encodelist.append(face_encode)
    return encodelist

In [7]:
encoded_list_known_faces=faceencodings(imgs)

In [8]:
import csv
from datetime import datetime

marked_names = set() 
current_date = datetime.now().strftime('%Y-%m-%d')

if os.path.exists('attendance.csv'):
    with open('attendance.csv', 'r') as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) >= 2 and row[1] == current_date:
                marked_names.add(row[0])

video = cv2.VideoCapture(0)

while True:
    suc, img = video.read()
    now = datetime.now()
    today_check = now.strftime('%Y-%m-%d')
    if today_check != current_date:
        marked_names.clear()
        current_date = today_check
        print(f"New day detected: {current_date}. Memory cleared.")

    img1 = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    face_loc1 = fr.face_locations(img1)
    faceencode1 = fr.face_encodings(img1, face_loc1)

    for enc_loc, faceloc in zip(faceencode1, face_loc1):
        matches = fr.compare_faces(encoded_list_known_faces, enc_loc)
        face_dis = fr.face_distance(encoded_list_known_faces, enc_loc)
        matchindex = np.argmin(face_dis)

        if matches[matchindex]:
            name = classnames[matchindex]
            if name not in marked_names:
                time_string = now.strftime('%H:%M:%S')

                
                with open('attendance.csv', 'a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow([name, current_date, time_string])

                marked_names.add(name) 
                print(f"Verified: {name} marked for {current_date}.")
        else:
            name = "unknown"

        y1, x2, y2, x1 = faceloc
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, name, (x1, y1-10), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2)

    cv2.imshow("Attendance System", img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video.release()
cv2.destroyAllWindows()

Verified: arjun_jayappan marked for 2026-02-20.
Verified: jomol_sabu marked for 2026-02-20.
Verified: anooja_giri marked for 2026-02-20.
